In [ ]:
# ChromaDB에 저장 후 유사 문장 검색
!pip install chromadb sentence-transformers

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path=".aa/bb/ccdb")
collection = client.get_or_create_collection(name="mytest",
                        metadata={'hnsw:space':'l2'})  # 'cosine', 'ip'

texts = [
    "Apple is a fruit",
    "Python is a programming language",
    "The sun rises in the east",
    "I love to eat mangoes",
]

ids = [str(i) for i in range(len(texts))]
print(ids)   # ['0', '1', '2', '3']

model = SentenceTransformer("all-MiniLM-L6-v2")
print(model.get_sentence_embedding_dimension())
print(model)
print(model.encode(texts).shape)  # (4, 384)

embeddings = model.encode(texts).tolist()
print(embeddings)

# vectordb에 저장
collection.add(documents=texts, ids=ids, embeddings=embeddings)

# vectordb의 자료 조회
record = collection.get(ids=["0"], include=['embeddings', 'documents'])
print('조회된 문서 : ', record['documents'][0])
print('벡터 (앞 10개) : ', record['embeddings'][0][:10])

In [ ]:
# vectordb의 자료 유사 문장 검색
query_data = "What is Python?"
query_vector = model.encode([query_data]).tolist()
print(query_vector)   # [[-0.06477224081754684, 0.036463022232055664, ...

result = collection.query(
    query_embeddings=query_vector,
    n_results = 2,
    include=['documents', 'distances']   # distances - query_vector와의 거리. 0에 근사하면 유사함
)

print('유사한 문장 검색 결과:')
for doc, dist in zip(result['documents'][0], result['distances'][0]):
    print(f'- 문장:{doc} (유사도거리:{dist:.4f})')
